# Exploration de l'API Lever (Postings API)

Objectif : comprendre ce que renvoie l'API Lever, et en quoi elle diffère de Greenhouse, avant d'écrire l'adaptateur.

Questions auxquelles ce notebook doit répondre :
1. Quelle est la structure de premier niveau de la réponse ? Où sont les offres ?
2. Quels champs correspondent au titre, au lieu, à la date, à l'identifiant, au lien ? Comment se comparent-ils à Greenhouse ?
3. Que renvoie l'API pour une entreprise inexistante ?
4. Que change le paramètre `mode=json` ?

Documentation : `github.com/lever/postings-api`

Pensez à faire **Clear All Outputs** avant chaque commit.

In [1]:
import json
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import httpx

## 0. Paramètres

Chez Lever, l'identifiant de l'entreprise s'appelle un **site** : c'est le segment après `jobs.lever.co/` dans l'adresse de la page carrières.

Un seul client pour tout le notebook : il réutilise la connexion et porte le timeout, comme on le fera en production.

In [2]:
SITE = "blablacar"
BASE_URL = "https://api.lever.co/v0/postings"
BASE_URL_EU = "https://api.eu.lever.co/v0/postings"

client = httpx.Client(timeout=30)

## 1. La requête

Remarquez qu'on ne passe aucune clé d'API : la lecture des offres publiées est publique.

In [3]:
response = client.get(f"{BASE_URL}/{SITE}", params={"mode": "json"})
print("Code de statut :", response.status_code)
print("Type de contenu :", response.headers.get("content-type"))
print("Taille :", round(len(response.content) / 1024), "Ko")
response.raise_for_status()

data = response.json()

Code de statut : 200
Type de contenu : application/json; charset=utf-8
Taille : 125 Ko


## 2. Question 1 : la structure de premier niveau

Chez Greenhouse, `data` était un dictionnaire et les offres se trouvaient sous la clé `"jobs"`. Et ici ?

In [4]:
print("Type de data :", type(data).__name__)

if isinstance(data, dict):
    print("Clés de premier niveau :", list(data.keys()))
else:
    print("Nombre d'éléments :", len(data))
    print("Type du premier élément :", type(data[0]).__name__)

Type de data : list
Nombre d'éléments : 7
Type du premier élément : dict


In [5]:
data

[{'additionalPlain': 'BlaBlaCar is an equal opportunity employer. We celebrate diversity and are committed to creating an inclusive environment for all employees.\n',
  'additional': '<div><em style="font-size: 16px;">BlaBlaCar is an equal opportunity employer. We celebrate diversity and are committed to creating an inclusive environment for all employees.</em></div>',
  'categories': {'commitment': 'Permanent',
   'department': 'General Administration',
   'location': 'Paris, France',
   'team': 'People',
   'allLocations': ['Paris, France']},
  'createdAt': 1789626911138,
  'descriptionPlain': "About BlaBlaCar\n\nBlaBlaCar is the world’s leading community-based travel app enabling 27 million members a year to carpool or travel by bus in 21 countries. Our team of 800 employees counts over 50 nationalities and is spread across our 5 global offices, 30% working fully remotely.\n\n\nWe are looking for a Compensation & Benefits Specialist with 5+ years of hands-on experience to support, e

**À vous** : notez ici votre constat. Quelle conséquence pour le code de votre adaptateur, comparé à `data["jobs"]` chez Greenhouse ?

Pour la suite du notebook, on appelle `offres` la liste des offres.

In [6]:
offres = data  # à vous d'expliquer pourquoi cette ligne suffit ici

## 3. Question 2 : anatomie d'une offre

Tous les champs de la première offre, avec leur type. Les valeurs longues sont tronquées.

In [7]:
premiere = offres[0]

for cle, valeur in premiere.items():
    apercu = repr(valeur)
    if len(apercu) > 90:
        apercu = apercu[:90] + "..."
    print(f"{cle:<24} {type(valeur).__name__:<6} {apercu}")

additionalPlain          str    'BlaBlaCar is an equal opportunity employer. We celebrate diversity and are committed to c...
additional               str    '<div><em style="font-size: 16px;">BlaBlaCar is an equal opportunity employer. We celebrat...
categories               dict   {'commitment': 'Permanent', 'department': 'General Administration', 'location': 'Paris, Fr...
createdAt                int    1789626911138
descriptionPlain         str    "About BlaBlaCar\n\nBlaBlaCar is the world’s leading community-based travel app enabling 2...
description              str    '<p><strong><span style="font-size: 18px;">About BlaBlaCar</span></strong><br><br><span st...
id                       str    'f9225e80-ee84-468b-bd73-897687bdb3c5'
lists                    list   [{'text': 'Your responsibilities', 'content': "<div>\n\n<li>Participate in designing, then...
text                     str    'Compensation & Benefits Specialist'
country                  str    'FR'
workplaceType        

### Les champs présents sur toutes les offres ?

Un champ peut exister sur une offre et manquer sur une autre (le salaire, par exemple). On compte, pour chaque champ, sur combien d'offres il apparaît.
Ça vous aidera à distinguer les champs obligatoires des optionnels pour le futur modèle `Offre`.

In [8]:
presence = Counter(cle for offre in offres for cle in offre)

print(f"Sur {len(offres)} offres :\n")
for cle, nombre in presence.most_common():
    print(f"{cle:<24} {nombre:>4}")

Sur 7 offres :

additionalPlain             7
additional                  7
categories                  7
createdAt                   7
descriptionPlain            7
description                 7
id                          7
lists                       7
text                        7
country                     7
workplaceType               7
opening                     7
openingPlain                7
descriptionBody             7
descriptionBodyPlain        7
hostedUrl                   7
applyUrl                    7


### Le champ `categories`

Le lieu, le type de contrat et l'équipe ne sont pas au premier niveau : ils sont regroupés dans un objet imbriqué.

In [9]:
print(json.dumps(premiere.get("categories"), indent=2, ensure_ascii=False))

{
  "commitment": "Permanent",
  "department": "General Administration",
  "location": "Paris, France",
  "team": "People",
  "allLocations": [
    "Paris, France"
  ]
}


### Les dates

La documentation ne mentionne pas de champ de date, mais la réponse réelle peut en contenir. On cherche les champs numériques dont le nom évoque une date.

Indice : un très grand entier qui représente une date est souvent un **timestamp**, c'est-à-dire un nombre de secondes (ou de millisecondes) écoulées depuis le 1er janvier 1970.

In [10]:
for cle, valeur in premiere.items():
    if isinstance(valeur, int) and ("At" in cle or "date" in cle.lower()):
        print(f"{cle} = {valeur}")
        print(
            "  interprété en secondes      :",
            datetime.fromtimestamp(valeur, tz=UTC) if valeur < 1e11 else "hors limites",
        )
        print("  interprété en millisecondes :", datetime.fromtimestamp(valeur / 1000, tz=UTC))

createdAt = 1789626911138
  interprété en secondes      : hors limites
  interprété en millisecondes : 2026-09-17 06:35:11.138000+00:00


**À vous** : quelle interprétation est la bonne ? Comparez avec le format de date de Greenhouse (`updated_at`). Votre futur modèle `Offre` devra convertir les deux vers un même type Python.

Et un champ de date non documenté : faut-il s'appuyer dessus ? Quel risque ?

### Les descriptions : HTML et texte brut

Chez Greenhouse, la description arrivait en HTML échappé, à décoder avec `html.unescape`. Lever fournit des variantes `...Plain`.

In [11]:
print("HTML  :", premiere.get("description", "")[:200])
print()
print("Plain :", premiere.get("descriptionPlain", "")[:200])
print()
print("Listes complémentaires :", [liste["text"] for liste in premiere.get("lists", [])])

HTML  : <p><strong><span style="font-size: 18px;">About BlaBlaCar</span></strong><br><br><span style="font-size: 16px;">BlaBlaCar is the world’s leading community-based travel app enabling 27 million members 

Plain : About BlaBlaCar

BlaBlaCar is the world’s leading community-based travel app enabling 27 million members a year to carpool or travel by bus in 21 countries. Our team of 800 employees counts over 50 na

Listes complémentaires : ['Your responsibilities', 'Your qualifications', 'What we have to offer', 'Interested in joining the ride?']


## 4. Vue d'ensemble

Titre, lieu, type de contrat, mode de travail. Comparez avec la section équivalente du notebook Greenhouse : quelles informations Lever donne-t-il en plus ?

In [12]:
for offre in offres[:15]:
    cat = offre.get("categories", {})
    print(
        f"{offre['text'][:45]:<45}  "
        f"{(cat.get('location') or '?')[:18]:<18}  "
        f"{(cat.get('commitment') or '?')[:12]:<12}  "
        f"{offre.get('workplaceType', '?')}"
    )

Compensation & Benefits Specialist             Paris, France       Permanent     hybrid
Confirmed Backend Engineer - Care              Paris, France       Permanent     hybrid
Confirmed Backend Engineer - Payment Squad     Paris, France       Permanent     onsite
Confirmed FP&A Analyst                         Paris, France       Permanent     hybrid
Product Manager - Discover                     Paris, France       Permanent     hybrid
Stagiaire Communication & Relations Publiques  Paris, France       Internship    hybrid
Strategy & Business Analyst Intern             Paris, France       Internship    hybrid


### Les valeurs réellement utilisées

Les filtres de l'API (`location`, `commitment`...) sont sensibles à la casse. Avant de s'en servir, il faut connaître les valeurs exactes.

In [13]:
for champ in ["location", "commitment", "team"]:
    valeurs = Counter(offre.get("categories", {}).get(champ) for offre in offres)
    print(f"{champ} :")
    for valeur, nombre in valeurs.most_common(8):
        print(f"   {nombre:>3}  {valeur}")
    print()

print("workplaceType :", Counter(offre.get("workplaceType") for offre in offres))

location :
     7  Paris, France

commitment :
     5  Permanent
     2  Internship

team :
     2  Back-end
     1  People
     1  Finance
     1  Product Management
     1  Corporate Communication
     1  Growth & Strategy

workplaceType : Counter({'hybrid': 6, 'onsite': 1})


## 5. Question 3 : une entreprise inexistante

Comparez avec la réponse de Greenhouse (section 8 de l'autre notebook).

In [14]:
reponse_erreur = client.get(f"{BASE_URL}/entreprise-qui-nexiste-pas", params={"mode": "json"})
print("Code de statut :", reponse_erreur.status_code)
print("Corps :", reponse_erreur.text[:200])

Code de statut : 404
Corps : {"ok":false,"error":"Document not found"}


## 6. Question 4 : à quoi sert `mode=json` ?

La documentation indique que le format dépend du paramètre `mode` **et** de l'en-tête HTTP `Accept`, le paramètre étant prioritaire. Vérifions.

In [15]:
sans_mode = client.get(f"{BASE_URL}/{SITE}")
print("Sans mode       :", sans_mode.status_code, sans_mode.headers.get("content-type"))
print("  début du corps :", sans_mode.text[:120].replace("\n", " "))
print()

avec_accept = client.get(f"{BASE_URL}/{SITE}", headers={"Accept": "application/json"})
print("Avec Accept JSON :", avec_accept.status_code, avec_accept.headers.get("content-type"))

Sans mode       : 200 application/json; charset=utf-8
  début du corps : [{"additionalPlain":"BlaBlaCar is an equal opportunity employer. We celebrate diversity and are committed to creating an

Avec Accept JSON : 200 application/json; charset=utf-8


**À vous** : quel risque si votre adaptateur oubliait ce paramètre ? Comment un test pourrait-il le détecter ?

## 7. La pagination : `skip` et `limit`

Contrairement à Greenhouse, Lever documente une pagination. Est-elle nécessaire pour récupérer toutes les offres ? On compare le nombre d'offres avec et sans `limit`.

In [16]:
page = client.get(f"{BASE_URL}/{SITE}", params={"mode": "json", "skip": 0, "limit": 5}).json()
print("Avec limit=5   :", len(page), "offres")
print("Sans limit     :", len(offres), "offres")
print("Mêmes premières offres ?", [o["id"] for o in page] == [o["id"] for o in offres[:5]])

Avec limit=5   : 5 offres
Sans limit     : 7 offres
Mêmes premières offres ? True


**À vous** : sans `limit`, l'API renvoie-t-elle tout ? Si oui, votre adaptateur peut-il ignorer la pagination pour l'instant ? (Pensez au principe : ne pas coder ce dont on n'a pas encore besoin.)

## 8. Bonus : l'instance européenne

Certaines entreprises sont hébergées sur `api.eu.lever.co`. Pour une entreprise donnée, interrogeons les deux instances.

In [17]:
SITES_A_TESTER = ["mistral", "qonto", "blablacar", "veepee", "swile"]

for site in SITES_A_TESTER:
    resultats = []
    for nom, base in [("global", BASE_URL), ("EU", BASE_URL_EU)]:
        r = client.get(f"{base}/{site}", params={"mode": "json"})
        nb = len(r.json()) if r.status_code == 200 else "-"
        resultats.append(f"{nom}: {r.status_code} ({nb} offres)")
    print(f"{site:<12}", " | ".join(resultats))

mistral      global: 200 (0 offres) | EU: 404 (- offres)
qonto        global: 200 (36 offres) | EU: 404 (- offres)
blablacar    global: 200 (7 offres) | EU: 404 (- offres)
veepee       global: 200 (103 offres) | EU: 404 (- offres)
swile        global: 200 (28 offres) | EU: 404 (- offres)


**À vous** : certaines entreprises répondent-elles sur une seule des deux instances ? Si oui, que devra faire votre adaptateur ? Notez-le pour plus tard, on ne le code pas tout de suite.

## 9. Enregistrer les fixtures

Point important : la fixture doit garder **exactement la même structure de premier niveau** que la vraie réponse. Sinon, vos tests vérifieraient un format qui n'existe pas en production.

On garde 3 offres par entreprise, pour des fichiers petits et lisibles. Pensez à changer `site` si une entreprise renvoie une erreur ou est sur l'instance EU.

In [ ]:
dossier_fixtures = Path("..") / "tests" / "fixtures"
dossier_fixtures.mkdir(parents=True, exist_ok=True)

for site in ["blablacar", "qonto"]:
    r = client.get(f"{BASE_URL}/{site}", params={"mode": "json"})
    r.raise_for_status()
    fixture = r.json()[:3]  # même structure que la réponse, réduite à 3 offres

    chemin = dossier_fixtures / f"lever_{site}.json"
    with open(chemin, "w", encoding="utf-8") as fichier:
        json.dump(fixture, fichier, ensure_ascii=False, indent=2)
    print("Fixture enregistrée :", chemin.resolve(), f"({len(fixture)} offres)")

Fixture enregistrée : C:\Users\User\Desktop\aicandidate\tests\fixtures\lever_blablacar.json (3 offres)
Fixture enregistrée : C:\Users\User\Desktop\aicandidate\tests\fixtures\lever_qonto.json (3 offres)
Fixture enregistrée : C:\Users\User\Desktop\aicandidate\tests\fixtures\lever_mistral.json (0 offres)


In [19]:
client.close()

## Synthèse : Greenhouse vs Lever

Remplissez ce tableau à partir de vos observations. Il servira directement à concevoir l'adaptateur Lever, puis le modèle `Offre` commun.

| | Greenhouse | Lever |
|---|---|---|
| URL | `boards-api.greenhouse.io/v1/boards/{token}/jobs` | |
| Structure de la réponse | dictionnaire, offres sous `"jobs"` | |
| Titre | `title` | |
| Lieu | `location.name` | |
| Date | `updated_at` (chaîne ISO 8601) | |
| Identifiant | `id` (entier) | |
| Lien de l'offre | `absolute_url` | |
| Description | `content` (HTML échappé) | |
| Entreprise inexistante | 404 | |
| Pagination | aucune | |
| Instance EU | `boards-api.eu.greenhouse.io` (à vérifier) | |